<a href="https://colab.research.google.com/github/lygitdata/GarmentIQ/blob/main/test/tutorial_landmark_refinement_and_derivation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial - GarmentIQ Landmark Refinement and Derivation

Detected landmarks often sit slightly off the garment edge, and some useful points are not
predicted by the model at all. **Refinement** snaps detected landmarks onto the true
boundary using a segmentation mask, while **derivation** computes entirely new landmarks
from geometric rules.

This tutorial shows how to run both steps, why each one needs a segmentation mask, and how
to plot detected and derived points together.

## Table of Contents

1. [Prerequisites](#prerequisites)
2. [Detect landmarks and a mask](#detect)
3. [Refine the landmarks](#refine)
4. [Derive new landmarks](#derive)

<a name="prerequisites"></a>
## Prerequisites

Install the package and download the test image and both models. On Colab you can keep
this section collapsed.

In [ ]:
# @title Install GarmentIQ
!pip install garmentiq -q

In [ ]:
# @title Import GarmentIQ and choose a device

import numpy as np
import torch

import garmentiq as giq
from garmentiq.landmark.detection.model_definition import PoseHighResolutionNet
from garmentiq.garment_classes import garment_classes
from garmentiq.segmentation.model_definition.birefnet import (
    BiRefNet,
    load_birefnet_config,
)

# GarmentIQ never grabs an accelerator on its own: every model loader and every
# inference function takes a `device` argument that defaults to "cpu". Pass it
# explicitly to use a GPU ("cuda") or Apple Silicon ("mps").
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Using device:", device)

In [ ]:
# @title Download the test image and the pretrained models

!mkdir -p ./test_image
!wget -q -O ./test_image/cloth_3.jpg \
    https://raw.githubusercontent.com/lygitdata/GarmentIQ/refs/heads/gh-pages/asset/img/cloth_3.jpg

# HRNet, for landmark detection
!mkdir -p ./models
!wget -q -O ./models/hrnet.pth \
    https://huggingface.co/lygitdata/garmentiq/resolve/main/hrnet.pth

# BiRefNet, for the segmentation mask both steps need
!mkdir -p ./models/birefnet
!wget -q -O ./models/birefnet/model.safetensors \
    https://huggingface.co/lygitdata/BiRefNet_garmentiq_backup/resolve/main/model.safetensors

print("Downloads finished.")

<a name="detect"></a>
## Detect landmarks and a mask

Refinement and derivation both work from a segmentation mask, so we run detection and
segmentation first. Detection is covered in the
[landmark detection tutorial](https://colab.research.google.com/github/lygitdata/GarmentIQ/blob/main/test/tutorial_landmark_detection.ipynb).

In [ ]:
HRNet = giq.landmark.detection.load_model(
    model_path="./models/hrnet.pth",
    model_class=PoseHighResolutionNet(),
    device=device,
)

coords, maxvals, detection_dict = giq.landmark.detect(
    class_name="vest dress",
    class_dict=garment_classes,
    image_path="./test_image/cloth_3.jpg",
    model=HRNet,
    scale_std=200.0,
    resize_dim=[288, 384],
    normalize_mean=[0.485, 0.456, 0.406],
    normalize_std=[0.229, 0.224, 0.225],
    device=device,
)

giq.landmark.plot(
    image_path="./test_image/cloth_3.jpg",
    coordinate=coords,
    figsize=(3, 3),
    color="green",
)

In [ ]:
birefnet = giq.segmentation.load_model(
    model_class=BiRefNet,
    model_path="./models/birefnet/model.safetensors",
    model_args=load_birefnet_config(),
    device=device,
)

original_img, mask = giq.segmentation.extract(
    model=birefnet,
    image_path="./test_image/cloth_3.jpg",
    resize_dim=(1024, 1024),
    normalize_mean=[0.485, 0.456, 0.406],
    normalize_std=[0.229, 0.224, 0.225],
    device=device,
)

giq.segmentation.plot(image_np=mask, figsize=(3, 3))

<a name="refine"></a>
## Refine the landmarks

`refine` searches a small window around each detected point and moves it onto the mask
boundary. `window_size` sets how far it may travel, and `ksize` and `sigmaX` control the
Gaussian blur applied to the mask before the search, which smooths away jagged edges.

In [ ]:
refined_coords, refined_detection_dict = giq.landmark.refine(
    class_name="vest dress",
    detection_np=coords,
    detection_conf=maxvals,
    detection_dict=detection_dict,
    mask=mask,
    window_size=5,
    ksize=(11, 11),
    sigmaX=0.0,
)

shift = np.linalg.norm(refined_coords[0] - coords[0], axis=1)
for i, d in enumerate(shift):
    print(f"landmark {i + 1:>2}: moved {d:6.2f} px")

In [ ]:
giq.landmark.plot(
    image_path="./test_image/cloth_3.jpg",
    coordinate=refined_coords,
    figsize=(3, 3),
    color="green",
)

<a name="derive"></a>
## Derive new landmarks

Some measurement points are not predicted by the model. They are marked
`predefined: False` in the garment class definition and computed instead, by intersecting
a geometric construction with the mask.

`derivation_dict` holds the rules. `derive` returns the newly derived coordinates and an
updated detection dictionary.

In [ ]:
derived_coords, derived_detection_dict = giq.landmark.derive(
    class_name="vest dress",
    detection_dict=refined_detection_dict,
    derivation_dict=giq.landmark.derivation_dict.derivation_dict,
    landmark_coords=refined_coords,
    np_mask=mask,
)

print("Derived landmarks:")
for landmark_id, xy in derived_coords.items():
    print(f"  id {landmark_id}: x={xy[0]:.1f}, y={xy[1]:.1f}")

Plot the refined landmarks together with a derived one to see where it
landed.

In [ ]:
derived_id = list(derived_coords.keys())[0]

giq.landmark.plot(
    image_path="./test_image/cloth_3.jpg",
    coordinate=np.concatenate(
        (refined_coords, np.array([[derived_coords[derived_id]]])), axis=1
    ),
    figsize=(3, 3),
    color="green",
)

Defining your own derivation rules is covered in the custom measurement
instruction advanced usage notebook.